# 02 - Análisis Descriptivo y Pruebas de Normalidad

Objetivo: evaluar la distribución de las variables continuas y resumir la muestra global con estadísticos descriptivos adecuados para variables no normales. El análisis bivariado por `alteracion_osea` se concentra en `03_bivariate_analysis.ipynb`.

In [17]:
from html import escape
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
from IPython.display import HTML, display
from scipy.stats import shapiro

warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.max_columns", None)

In [18]:
PROCESSED_DATA_DIR = Path(
    "/home/marcos-maravilla/análisis_estadístico_osteoporosis/data/processed"
)
PARQUET_PATH = PROCESSED_DATA_DIR / "BD_Clean_Osteoporosis.parquet"
PICKLE_PATH = PROCESSED_DATA_DIR / "BD_Clean_Osteoporosis.pkl"

try:
    df_clean = pd.read_parquet(PARQUET_PATH)
    loaded_path = PARQUET_PATH
except Exception:
    df_clean = pd.read_pickle(PICKLE_PATH)
    loaded_path = PICKLE_PATH

print(f"Dataset cargado desde: {loaded_path}")
print(f"Dimensiones: {df_clean.shape[0]:,} filas x {df_clean.shape[1]:,} columnas")

Dataset cargado desde: /home/marcos-maravilla/análisis_estadístico_osteoporosis/data/processed/BD_Clean_Osteoporosis.parquet
Dimensiones: 405 filas x 75 columnas


In [19]:
TABLE_CSS = """
<style>
.report-table-wrap{max-width:920px;overflow-x:auto;background:#fff!important;color:#111!important;color-scheme:light;padding:18px 20px;border:1px solid #d7d2c3;box-shadow:0 2px 8px rgba(0,0,0,.16);font-family:'Times New Roman',Times,serif;}
.report-table-wrap *{box-sizing:border-box;color:#111!important;opacity:1!important;}
.report-title{background:#fff!important;font-size:15px;line-height:1.35;margin:0 0 10px 0;}
.report-title strong{font-style:italic;}
.report-note{background:#fff!important;font-size:12.5px;line-height:1.35;margin:10px 0 0 0;}
.report-table{border-collapse:collapse;min-width:720px;width:100%;background:#fff!important;font-size:13.5px;line-height:1.3;table-layout:auto;}
.report-table th{background:#f3f0e8!important;border-top:2px solid #111;border-bottom:1.5px solid #111;padding:8px 10px;text-align:center;font-weight:700;vertical-align:middle;}
.report-table td{background:#fff!important;border-bottom:1px solid #222;padding:6px 10px;vertical-align:middle;}
.report-table td:first-child{font-weight:700;background:#fbfaf6!important;}
.report-table .num{text-align:center;white-space:nowrap;}
.report-table .text{text-align:left;}
</style>
"""

def render_report_table(table, title, note=None):
    html = [TABLE_CSS, "<div class='report-table-wrap'>"]
    html.append(f"<p class='report-title'><strong>{escape(title)}</strong></p>")
    html.append("<table class='report-table'><thead><tr>")

    for column in table.columns:
        html.append(f"<th>{escape(str(column))}</th>")
    html.append("</tr></thead><tbody>")

    for _, row in table.iterrows():
        html.append("<tr>")
        for value in row:
            css_class = "num" if isinstance(value, (int, float, np.integer, np.floating)) else "text"
            html.append(f"<td class='{css_class}'>{escape(str(value))}</td>")
        html.append("</tr>")

    html.append("</tbody></table>")
    if note:
        html.append(f"<p class='report-note'>{escape(note)}</p>")
    html.append("</div>")
    return HTML("".join(html))

def format_p_value(value):
    return "< 0.001" if value < 0.001 else f"{value:.4f}"

## 1. Pruebas de Normalidad (Shapiro-Wilk)

In [20]:
continuous_variables = ["edad", "peso_(kg)", "altura_(cm)", "imc"]
variable_labels = {
    "edad": "Edad (años)",
    "peso_(kg)": "Peso (kg)",
    "altura_(cm)": "Altura (cm)",
    "imc": "IMC",
}

normality_results = []

for variable in continuous_variables:
    values = pd.to_numeric(df_clean[variable], errors="coerce").dropna()
    statistic, p_value = shapiro(values)
    normality_results.append(
        {
            "Variable": variable_labels[variable],
            "n": len(values),
            "W": f"{statistic:.4f}",
            "p-value": format_p_value(p_value),
            "Interpretación": "No normal" if p_value < 0.05 else "Normal",
        }
    )

normality_results = pd.DataFrame(normality_results)
display(render_report_table(normality_results, "Tabla 1. Pruebas de normalidad para variables continuas", "Prueba de Shapiro-Wilk; p < 0.05 indica desviación estadísticamente significativa de la normalidad."))

Variable,n,W,p-value,Interpretación
Edad (años),405,0.9770,< 0.001,No normal
Peso (kg),405,0.9850,< 0.001,No normal
Altura (cm),405,0.9714,< 0.001,No normal
IMC,405,0.9792,< 0.001,No normal


## 2. Estadística Descriptiva de Variables Continuas

In [21]:
descriptive_continuous = []

for variable in continuous_variables:
    values = pd.to_numeric(df_clean[variable], errors="coerce").dropna()
    q1 = values.quantile(0.25)
    median = values.median()
    q3 = values.quantile(0.75)
    descriptive_continuous.append(
        {
            "Variable": variable_labels[variable],
            "n": len(values),
            "Mediana (Q1, Q3)": f"{median:.2f} ({q1:.2f}, {q3:.2f})",
            "RIC": f"{q1:.2f} - {q3:.2f}",
            "Mínimo - Máximo": f"{values.min():.2f} - {values.max():.2f}",
            "Valores nulos": int(df_clean[variable].isna().sum()),
        }
    )

descriptive_continuous = pd.DataFrame(descriptive_continuous)
display(render_report_table(descriptive_continuous, "Tabla 2. Descripción global de variables continuas", "Las variables se resumen con mediana y rango intercuartílico por la evidencia de no normalidad."))

Variable,n,"Mediana (Q1, Q3)",RIC,Mínimo - Máximo,Valores nulos
Edad (años),405,"65.00 (59.00, 72.00)",59.00 - 72.00,50.00 - 90.00,0
Peso (kg),405,"69.00 (62.00, 77.00)",62.00 - 77.00,38.00 - 112.00,0
Altura (cm),405,"155.00 (151.00, 162.00)",151.00 - 162.00,134.00 - 188.00,0
IMC,405,"28.19 (25.28, 31.23)",25.28 - 31.23,17.35 - 48.68,0
